# 03 — Data Quality Framework

Ce notebook évalue la qualité du dataset final `final_data_ai_jobs_clean_2020_2026.csv` avant son utilisation dans les étapes suivantes du projet : stockage NoSQL, Web Content Mining, Graph Mining, prédiction et dashboard.

Objectifs :

- vérifier la complétude des colonnes ;
- analyser les valeurs manquantes ;
- détecter les doublons ;
- étudier les déséquilibres par source, année et pays ;
- documenter les limites du dataset ;
- créer une version réduite du dataset en supprimant les colonnes trop incomplètes : `company_name`, `job_type`, `job_url`, `description`.


## 1. Importation des bibliothèques

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

## 2. Chargement du dataset final clean

In [4]:
df = pd.read_csv("final_data_ai_jobs_clean_2020_2026.csv")

print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])

print("Colonnes :")
print(df.columns.tolist())

df.head(3).T

Nombre de lignes : 751801
Nombre de colonnes : 30
Colonnes :
['source', 'platform', 'job_id', 'job_title', 'company_name', 'country', 'city', 'date_posted', 'year', 'month', 'year_month', 'job_type', 'remote_status', 'experience_level', 'education_required', 'industry', 'category', 'description', 'skills', 'tools_used', 'salary', 'salary_currency', 'job_url', 'original_file', 'country_clean', 'skills_clean', 'skills_original', 'remote_status_clean', 'remote_status_original', 'job_category_clean']


,0,1,2
source,global_ai_jobs_dataset,global_ai_jobs_dataset,global_ai_jobs_dataset
platform,Synthetic / Global,Synthetic / Global,Synthetic / Global
job_id,1,2,3
job_title,AI Researcher,MLOps Engineer,Data Analyst
company_name,NaN,NaN,NaN
country,Canada,India,United Kingdom
city,Berlin,Tokyo,Bangalore
date_posted,2021-04-01 00:00:00,2020-04-12 00:00:00,2023-01-31 00:00:00
year,2021,2020,2023
month,4.0,4.0,1.0


## 3. Conversion et vérification des dates

In [6]:
df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")

print("Date minimale :", df["date_posted"].min())
print("Date maximale :", df["date_posted"].max())
print("Dates manquantes :", df["date_posted"].isna().sum())

print("Offres par année :")
print(df["year"].value_counts().sort_index())

Date minimale : 2020-01-01 00:00:00
Date maximale : 2026-05-06 16:47:40
Dates manquantes : 1240
Offres par année :
year
2020     20010
2021     20175
2022     19872
2023    641784
2024     27642
2025     20985
2026      1333
Name: count, dtype: int64


## 4. Analyse des valeurs manquantes

Cette étape permet d'identifier les colonnes fiables et les colonnes moins exploitables. Les colonnes très incomplètes ne seront pas forcément supprimées immédiatement du fichier original, mais une version réduite sera créée plus loin.

In [7]:
missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": round(df.isna().mean() * 100, 2)
}).sort_values("missing_count", ascending=False)

missing_table

,missing_count,missing_percent
description,748740,99.59
job_url,743008,98.83
industry,630860,83.91
salary_currency,630308,83.84
salary,610543,81.21
job_type,130543,17.36
company_name,120030,15.97
education_required,9698,1.29
tools_used,8793,1.17
experience_level,1631,0.22


## 5. Qualité des colonnes essentielles

Les colonnes essentielles pour le projet sont :

- `job_title` : titre de l'offre ;
- `country` : pays ;
- `date_posted` : date de publication ;
- `skills` : compétences ;
- `source`, `platform`, `original_file` : traçabilité des données.


In [9]:
essential_cols = ["job_title", "country", "date_posted", "skills", "source", "platform", "original_file"]

essential_quality = pd.DataFrame({
    "missing_count": df[essential_cols].isna().sum(),
    "missing_percent": round(df[essential_cols].isna().mean() * 100, 2),
    "non_null_count": df[essential_cols].notna().sum()
})

essential_quality

,missing_count,missing_percent,non_null_count
job_title,0,0.00,751801
country,265,0.04,751536
date_posted,1240,0.16,750561
skills,325,0.04,751476
source,0,0.00,751801
platform,0,0.00,751801
original_file,0,0.00,751801


## 6. Suppression des colonnes très incomplètes

Les colonnes suivantes contiennent beaucoup de valeurs manquantes ou ne sont pas indispensables pour les analyses principales :

- `company_name`
- `job_type`
- `job_url`
- `description`

On crée donc une version réduite du dataset sans ces colonnes.  
Important : on garde le fichier original `final_data_ai_jobs_clean_2020_2026.csv` intact, et on sauvegarde une nouvelle version.


In [13]:
cols_to_drop = ["company_name", "job_type", "job_url", "description", "industry", "salary_currency", "salary"]

# Vérifier que les colonnes existent avant suppression
existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
print("Colonnes à supprimer :", existing_cols_to_drop)

# Créer une copie réduite
df_reduced = df.drop(columns=existing_cols_to_drop).copy()

print("Shape avant suppression :", df.shape)
print("Shape après suppression :", df_reduced.shape)

print("Colonnes restantes :")
print(df_reduced.columns.tolist())

Colonnes à supprimer : ['company_name', 'job_type', 'job_url', 'description', 'industry', 'salary_currency', 'salary']
Shape avant suppression : (751801, 30)
Shape après suppression : (751801, 23)
Colonnes restantes :
['source', 'platform', 'job_id', 'job_title', 'country', 'city', 'date_posted', 'year', 'month', 'year_month', 'remote_status', 'experience_level', 'education_required', 'category', 'skills', 'tools_used', 'original_file', 'country_clean', 'skills_clean', 'skills_original', 'remote_status_clean', 'remote_status_original', 'job_category_clean']


## 7. Vérification des valeurs manquantes après suppression

In [14]:
missing_reduced = pd.DataFrame({
    "missing_count": df_reduced.isna().sum(),
    "missing_percent": round(df_reduced.isna().mean() * 100, 2)
}).sort_values("missing_count", ascending=False)

missing_reduced

,missing_count,missing_percent
education_required,9698,1.29
tools_used,8793,1.17
experience_level,1631,0.22
city,1246,0.17
date_posted,1240,0.16
remote_status_original,1206,0.16
skills_clean,325,0.04
skills,325,0.04
country,265,0.04
country_clean,265,0.04


## 8. Sauvegarde du dataset réduit

Ce fichier peut être utilisé pour les analyses où les colonnes très incomplètes ne sont pas nécessaires.


In [15]:
df_reduced.to_csv(
    "final_data_ai_jobs_clean_reduced_2020_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichier sauvegardé : final_data_ai_jobs_clean_reduced_2020_2026.csv")
print("Shape :", df_reduced.shape)

Fichier sauvegardé : final_data_ai_jobs_clean_reduced_2020_2026.csv
Shape : (751801, 23)


## 9. Analyse des doublons

In [16]:
print("Doublons exacts dans df :", df.duplicated().sum())
print("Doublons exacts dans df_reduced :", df_reduced.duplicated().sum())

# Doublons potentiels basés sur les colonnes disponibles après suppression
potential_duplicate_cols = ["job_title", "country", "date_posted", "source"]

potential_duplicates = df_reduced.duplicated(
    subset=potential_duplicate_cols,
    keep=False
).sum()

print("Doublons potentiels sur", potential_duplicate_cols, ":", potential_duplicates)

Doublons exacts dans df : 0
Doublons exacts dans df_reduced : 0
Doublons potentiels sur ['job_title', 'country', 'date_posted', 'source'] : 77628


## 10. Analyse du déséquilibre par source

In [17]:
source_distribution = df_reduced["original_file"].value_counts().reset_index()
source_distribution.columns = ["original_file", "number_of_jobs"]
source_distribution["percentage"] = round(source_distribution["number_of_jobs"] / len(df_reduced) * 100, 2)

source_distribution

,original_file,number_of_jobs,percentage
0,huggingface_global_2023_data_jobs_skills_filte...,622067,82.74
1,global_ai_jobs_dataset.csv,120000,15.96
2,linkedin_us_uk_canada_australia_2024-01-12_to_...,5732,0.76
3,linkedin_datastax_2023-12-05_to_2024-04-20_dat...,1777,0.24
4,jobs_raw.json,960,0.13
5,kaggle_global_2025_data_science_jobs_skills_fi...,941,0.13
6,jobs_data_ai_clean.csv,274,0.04
7,aijobs_raw.csv,50,0.01


## 11. Analyse du déséquilibre temporel

In [18]:
year_distribution = df_reduced["year"].value_counts().sort_index().reset_index()
year_distribution.columns = ["year", "number_of_jobs"]
year_distribution["percentage"] = round(year_distribution["number_of_jobs"] / len(df_reduced) * 100, 2)

year_distribution

,year,number_of_jobs,percentage
0,2020,20010,2.66
1,2021,20175,2.68
2,2022,19872,2.64
3,2023,641784,85.37
4,2024,27642,3.68
5,2025,20985,2.79
6,2026,1333,0.18


## 12. Analyse du déséquilibre géographique

In [19]:
country_distribution = df_reduced["country"].value_counts(dropna=False).reset_index()
country_distribution.columns = ["country", "number_of_jobs"]
country_distribution["percentage"] = round(country_distribution["number_of_jobs"] / len(df_reduced) * 100, 2)

country_distribution.head(30)

,country,number_of_jobs,percentage
0,United States,199743,26.57
1,India,56012,7.45
2,United Kingdom,46123,6.14
3,Germany,33488,4.45
4,France,32273,4.29
5,Singapore,31216,4.15
6,Netherlands,28268,3.76
7,Canada,25601,3.41
8,Australia,21516,2.86
9,Spain,20278,2.70


## 13. Focus Maroc

In [21]:
df_morocco = df_reduced[df_reduced["country"] == "Morocco"]

print("Nombre d'offres au Maroc :", df_morocco.shape[0])

print("Offres Maroc par année :")
print(df_morocco["year"].value_counts().sort_index())

print("Sources des offres Maroc :")
print(df_morocco["original_file"].value_counts())

print("Top titres Maroc :")
print(df_morocco["job_title"].value_counts().head(20))

Nombre d'offres au Maroc : 1058
Offres Maroc par année :
year
2023    909
2025      3
2026    146
Name: count, dtype: int64
Sources des offres Maroc :
original_file
huggingface_global_2023_data_jobs_skills_filtered.csv    909
jobs_raw.json                                             75
jobs_data_ai_clean.csv                                    74
Name: count, dtype: int64
Top titres Maroc :
job_title
Data Engineer                                                                     84
Data Scientist                                                                    42
Data Analyst                                                                      39
Senior Data Engineer                                                              23
Senior Data Scientist                                                             21
Data scientist                                                                    18
Data Engineer (H/F)                                                               16
Se

## 14. Qualité des skills

In [22]:
print("Offres avec skills :", df_reduced["skills"].notna().sum())
print("Offres sans skills :", df_reduced["skills"].isna().sum())
print("Pourcentage avec skills :", round(df_reduced["skills"].notna().mean() * 100, 2), "%")

# Nombre moyen de skills par offre
num_skills = df_reduced["skills"].apply(lambda x: len(str(x).split(",")) if pd.notna(x) else 0)

print("Nombre moyen de skills par offre :", round(num_skills.mean(), 2))
print("Nombre médian de skills par offre :", num_skills.median())

Offres avec skills : 751476
Offres sans skills : 325
Pourcentage avec skills : 99.96 %
Nombre moyen de skills par offre : 5.31
Nombre médian de skills par offre : 4.0


## 15. Qualité du remote_status

In [23]:
remote_quality = df_reduced["remote_status"].value_counts(dropna=False).reset_index()
remote_quality.columns = ["remote_status", "number_of_jobs"]
remote_quality["percentage"] = round(remote_quality["number_of_jobs"] / len(df_reduced) * 100, 2)

remote_quality

,remote_status,number_of_jobs,percentage
0,On-site,608492,80.94
1,Remote,100902,13.42
2,Hybrid,40077,5.33
3,Not specified,2330,0.31


## 16. Tableau synthétique du Data Quality Framework

In [24]:
quality_summary = pd.DataFrame({
    "dimension": [
        "Titres des postes",
        "Pays",
        "Dates",
        "Compétences",
        "Sources",
        "Remote status",
        "Descriptions",
        "Salaires",
        "Équilibre temporel",
        "Équilibre géographique",
        "Équilibre des sources"
    ],
    "evaluation": [
        "Très bonne",
        "Bonne",
        "Très bonne",
        "Très bonne",
        "Très bonne",
        "Bonne",
        "Faible dans le fichier original",
        "Faible",
        "Déséquilibré",
        "Déséquilibré",
        "Déséquilibré"
    ],
    "commentaire": [
        "La colonne job_title est complète.",
        "Très peu de pays manquants après nettoyage.",
        "Très peu de dates manquantes.",
        "La grande majorité des offres contient des skills.",
        "La source et le fichier d'origine sont disponibles pour toutes les lignes.",
        "Remote status normalisé en Remote, Hybrid, On-site et Not specified.",
        "La colonne description a été supprimée de la version réduite car très incomplète.",
        "La colonne salaire reste très incomplète.",
        "L'année 2023 est dominante à cause du volume Hugging Face.",
        "Les États-Unis et les grands pays anglophones sont fortement représentés.",
        "Certaines sources contiennent beaucoup plus de lignes que d'autres."
    ]
})

quality_summary

,dimension,evaluation,commentaire
0,Titres des postes,Très bonne,La colonne job_title est complète.
1,Pays,Bonne,Très peu de pays manquants après nettoyage.
2,Dates,Très bonne,Très peu de dates manquantes.
3,Compétences,Très bonne,La grande majorité des offres contient des ski...
4,Sources,Très bonne,La source et le fichier d'origine sont disponi...
5,Remote status,Bonne,"Remote status normalisé en Remote, Hybrid, On-..."
6,Descriptions,Faible dans le fichier original,La colonne description a été supprimée de la v...
7,Salaires,Faible,La colonne salaire reste très incomplète.
8,Équilibre temporel,Déséquilibré,L'année 2023 est dominante à cause du volume H...
9,Équilibre géographique,Déséquilibré,Les États-Unis et les grands pays anglophones ...


## 17. Sauvegarde des tableaux de qualité

In [25]:
missing_table.to_csv("quality_missing_values_original.csv", encoding="utf-8-sig")
missing_reduced.to_csv("quality_missing_values_reduced.csv", encoding="utf-8-sig")
source_distribution.to_csv("quality_source_distribution.csv", index=False, encoding="utf-8-sig")
year_distribution.to_csv("quality_year_distribution.csv", index=False, encoding="utf-8-sig")
country_distribution.to_csv("quality_country_distribution.csv", index=False, encoding="utf-8-sig")
quality_summary.to_csv("quality_framework_summary.csv", index=False, encoding="utf-8-sig")

print("Tableaux de qualité sauvegardés.")

Tableaux de qualité sauvegardés.


## 18. Conclusion

Le Data Quality Framework montre que le dataset est exploitable pour le projet, car les colonnes essentielles (`job_title`, `country`, `date_posted`, `skills`, `source`, `platform`, `original_file`) sont globalement complètes.

Cependant, plusieurs limites doivent être mentionnées :

- le dataset est déséquilibré par source, avec une forte dominance de Hugging Face 2023 ;
- le dataset est déséquilibré temporellement, surtout autour de l'année 2023 ;
- le dataset est déséquilibré géographiquement, avec une forte présence des États-Unis ;
- certaines colonnes comme `salary`, `description`, `job_url`, `job_type` et `company_name` sont incomplètes ou peu utiles pour l'analyse globale.

Pour cette raison, une version réduite `final_data_ai_jobs_clean_reduced_2020_2026.csv` a été créée en supprimant les colonnes les plus problématiques : `company_name`, `job_type`, `job_url` et `description`.
